In [0]:
%run ../../00_common/data_utils

In [0]:
def delete_expired_backup():
    """
    删除 t_cbr_dataset_backup 中已过期的备份数据。
    过期判定：current_timestamp() > expired_time。
    """
    target_db = get_env_config('silver_mdm_anonymization_database')
    backup_table = f"{target_db}.t_cbr_dataset_backup"
    expired_condition = "current_timestamp() > expired_time"

    expired_count = spark.table(backup_table).where(expired_condition).count()

    if expired_count == 0:
        print("nothing to delete")
        return

    DeltaTable.forName(spark, backup_table).delete(expired_condition)
    print(f"deleted {expired_count} expired backup records")

In [0]:
def hash_and_complete(task_id):
    """
    对 t_mdm_anonymization_log 表中当前 task_id 对应记录的 ConsumerId 进行 AES 加密处理。
    可逆，解密需使用相同密钥配合 aes_decrypt()。
    """
    target_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{target_db}.t_mdm_anonymization_log"
    condition = (F.col("status") == ANON_STATUS_IN_PROGRESS) & (F.col("task_id") == task_id)

    record_count = spark.table(log_table).where(condition).count()

    if record_count == 0:
        print("nothing to hash")
        return

    secret_key = get_env_config('anonymization_aes_key')
    # ponytail: 一次 update 同时加密 + 标记 Complete，原子化避免重复加密
    DeltaTable.forName(spark, log_table).update(
        condition,
        {
            "ConsumerId": F.hex(F.aes_encrypt(F.col("ConsumerId"), F.lit(secret_key))),
            "status": F.lit(ANON_STATUS_COMPLETE)
        }
    )
    print(f"hashed {record_count} ConsumerId values")

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

step_name = "delete_expired_backup"
step_num = "04"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    # 清理过期备份
    delete_expired_backup()
    # 对 ConsumerId 做可逆 AES 加密，同时标记 Complete
    hash_and_complete(task_id)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )